# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q1 = q('''
SELECT t.title, a.name, a.country
FROM tracks t 
JOIN artists a
    ON t.artist_id = a.artist_id
''')
q1

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


Explanation: I joined the artists table on the tracks table using artist_id which allowed me to create a table where an artist's song matched up with their name and country. I also ensured there were 9 rows.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q2 = q('''
SELECT genre, AVG(seconds) as averageSeconds
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY averageSeconds DESC
LIMIT 1
''')
q2

,genre,averageSeconds
0,Electronic,287.5


Explanation: I used averageSeconds to track the average track length and then selected the genre with the highest average track length using averageSeconds. I found that Electronic music had the longest average track length of 287.5 seconds. I also made sure to ensure genre wasn't null.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q3 = q('''
SELECT user, 
    COUNT(*) AS plays,
    COUNT(DISTINCT track_id) as distinct_tracks 
FROM plays
GROUP BY user
''')
q3

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


Explanation: I created this table by taking a user's total plays and then counting how many distinct tracks they listened to, putting both these numbers by their name to reveal how many plays and how many distinct tracks they each had. My table revealed that every user's plays equalled their distinct tracks.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q4 = q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p
    ON t.track_id = p.track_id 
WHERE p.track_id IS NULL
''')
q4

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


Explanation: I used LEFT JOIN so that every track would be available even if there were no matching plays and then selected each track where its plays were null. My table revealed that only Ridgeline and Untitled Demo haven't been played.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q5 = q('''
SELECT a.name,
    SUM(t.seconds) as totalSeconds,
    ROUND(SUM(t.seconds) / 60.0, 1) AS totalMinutes 
FROM plays p
JOIN tracks t
    ON p.track_id = t.track_id
JOIN artists a
    ON t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY totalSeconds DESC
''')
q5

,name,totalSeconds,totalMinutes
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


Explanation: I started by finding the total listening time in seconds for each artist and then converted those seconds to minutes. I then joined the plays, tracks, and artists table and ordered the table by total time descending. My table revealed that Kestrel had the most listening time while Marisol had the least.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q6 = q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL
''')
q6
#WHERE genre != 'Pop' would have resulted in this row not showing because NULL is excluded from these checks and must be explicitly found.

,track_id,title
0,18,Untitled Demo


Explanation: I selected all tracks from the track table where genre was null, using an explicit check to ensure only null entries were returned. My table revealed that only Untitled Demo has a null genre.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q7 = q('''
SELECT played_on,
    COUNT(*) AS plays,
    COUNT(DISTINCT user) as distinctUsers
FROM plays
GROUP BY played_on
ORDER BY played_on
''')
q7

,played_on,plays,distinctUsers
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


Explanation: I first collected the number of plays and distinct users per played_on entry and then grouped and ordered by played_on. My table correctly showed the played_on dates in ascending order with the plays and distinct users per date.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

Query 5 gave me the most trouble because it involved juggling the most tables and factors compared to the others. A specific misunderstanding I had was dividing the total seconds by 60 instead of 60.0, which resulted in the displayed minutes not having the correct decimals. After changing to 60.0 my minutes correctly displayed the expected decimal values and I was able to correctly complete the query.